In [1]:
import copy
import json
import numbers
import os
import sys
import time
from datetime import datetime, timezone
from functools import partial
from pathlib import Path
from types import SimpleNamespace

import kagglehub
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import scipy.sparse as sp
import torch
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "lightgcl.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from lightgcl import LightGCL, LightGCLSoftmax  # noqa: E402
from utils import normalize_adj, svd_view, to_torch_sparse, topk_metrics, train_epoch  # noqa: E402

DATA_DIR = Path(
    os.environ.get("EXPEDIA_DATA_DIR")
    or kagglehub.competition_download("expedia-hotel-recommendations")
)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def fmt_num(x, digits=4):
    """37670293 -> '37.670.293', 0.12345 -> '0,1235' (at most `digits` decimals)."""
    text = f"{x:,.0f}" if float(x).is_integer() else f"{x:,.{digits}f}".rstrip("0").rstrip(".")
    return text.translate(str.maketrans(",.", ".,"))


def fmt_pct(x, digits=2):
    """0.359036 -> '35,90%'."""
    return f"{x:.{digits}%}".replace(".", ",")


def fmt_table(df):
    """Display copy of df: numbers in Vietnamese format, columns whose name contains '%' as percentages."""
    def cell(value, pct):
        if isinstance(value, bool) or not isinstance(value, numbers.Number):
            return value
        if pd.isna(value):
            return "—"
        return fmt_pct(value) if pct else fmt_num(value)

    out = df.astype(object)
    for col in df.columns:
        out[col] = [cell(value, "%" in str(col)) for value in df[col]]
    return out

In [2]:
cfg = SimpleNamespace(
    item_cols=["hotel_cluster"],
    val_start="2014-09-01",
    test_start="2014-11-01",
    test_end="2015-01-01",
    embed_dim=64,
    n_layers=2,
    temp=0.2,
    lambda_1=0.2,
    lambda_2=1e-7,
    dropout=0.0,
    lr=1e-4,
    batch_size=1024,
    epochs=20,
    patience=3,
    main_metric="recall@5",
    topks=(5, 10, 20),
    seed=2026,
    retrain=False,  # True: train every variant again (about 2.5 hours on an RTX 4080) and evaluate the new checkpoints
    variants={  # checkpoint = (run directory in data/runs, key in its model.pt) of the best validation epoch
        "Có negative sampling, q=5": {"loss": "bpr", "svd_q": 5, "svd_exact": False,
                                      "checkpoint": ("lightgcl_20260930_094833", "bpr")},
        "Không negative sampling, q=5": {"loss": "softmax", "svd_q": 5, "svd_exact": False,
                                         "checkpoint": ("lightgcl_20260930_094833", "softmax")},
        "Có negative sampling, q=2": {"loss": "bpr", "svd_q": 2, "svd_exact": True,
                                      "checkpoint": ("lightgcl_20260930_112653", "bpr-q2-exact")},
        "Không negative sampling, q=2": {"loss": "softmax", "svd_q": 2, "svd_exact": True,
                                         "checkpoint": ("lightgcl_20260930_112653", "softmax-q2-exact")},
    },
)

In [3]:
COLUMN_TYPES = {"date_time": pa.timestamp("s"), "user_id": pa.int32(), "is_booking": pa.int8(),
                **{c: pa.int32() for c in cfg.item_cols}}
events = pv.read_csv(
    DATA_DIR / "train.csv",
    convert_options=pv.ConvertOptions(include_columns=list(COLUMN_TYPES), column_types=COLUMN_TYPES),
).to_pandas()
val_start, test_start, test_end = (pd.Timestamp(d) for d in (cfg.val_start, cfg.test_start, cfg.test_end))
in_train = (events["date_time"] < val_start) & (events["is_booking"] == 1)
in_eval = (events["is_booking"] == 1) & events["date_time"].between(val_start, test_end, inclusive="left")
data = events.loc[in_train | in_eval, ["date_time", "user_id", *cfg.item_cols]].copy()
data["is_train"] = in_train[in_train | in_eval].to_numpy()
del events
data["item"] = data.groupby(cfg.item_cols, sort=True).ngroup().astype(np.int32)
n_items = data["item"].nunique()

train = data[data["is_train"]]
user_index = pd.Index(np.sort(train["user_id"].unique()), name="user_id")
n_users = len(user_index)
pair_counts = train.groupby([user_index.get_indexer(train["user_id"]), train["item"]]).size()
train_counts = sp.csr_matrix(
    (pair_counts.to_numpy(np.float32),
     (pair_counts.index.get_level_values(0), pair_counts.index.get_level_values(1))),
    shape=(n_users, n_items),
)
train_csr = (train_counts > 0).astype(np.float32).tocsr()



def make_split(start, end):
    """Labels of the bookings in [start, end): one row per user, one column per cluster."""
    pairs = data[~data["is_train"] & data["date_time"].between(start, end, inclusive="left")][["user_id", "item"]]
    pairs = pairs.drop_duplicates()
    users, row = np.unique(pairs["user_id"].to_numpy(), return_inverse=True)
    return SimpleNamespace(
        uidx=user_index.get_indexer(users),  # -1 for users with no edge in the train graph
        labels=sp.csr_matrix((np.ones(len(pairs), np.float32), (row, pairs["item"].to_numpy())),
                             shape=(len(users), n_items)),
    )


val_split = make_split(val_start, test_start)
test_split = make_split(test_start, test_end)
print(f"Graph train: {fmt_num(n_users)} user × {fmt_num(n_items)} cluster, {fmt_num(train_csr.nnz)} cạnh; "
      f"validation: {fmt_num(len(val_split.uidx))} user, test: {fmt_num(len(test_split.uidx))} user, "
      f"{fmt_pct((test_split.uidx >= 0).mean())} user test có trong graph train")

Graph train: 664.727 user × 100 cluster, 1.776.919 cạnh; validation: 226.277 user, test: 222.688 user, 60,89% user test có trong graph train


## 1. Đánh giá

In [4]:
METRICS = [f"{m}@{k}" for m in ("recall", "ndcg") for k in cfg.topks]
EVAL_BATCH = max(256, 2**24 // n_items)
item_pop = torch.as_tensor(np.asarray(train_csr.sum(0)).ravel(), device=DEVICE)


def to_dense(csr_rows):
    return torch.as_tensor(csr_rows.toarray(), device=DEVICE)


def pop_scores(uidx):
    return item_pop.expand(len(uidx), -1)


def history_scores(uidx):
    # Booking count first, popularity (scaled below 1) only breaks ties.
    return to_dense(train_counts[uidx.cpu().numpy()]) + item_pop / (item_pop.max() + 1)


@torch.no_grad()
def evaluate(score_fn, split):
    """Mean metrics over the users of `split` that are in the train graph; score_fn maps graph user
    indices to a (users x items) score tensor."""
    warm = split.uidx >= 0
    uidx, labels = split.uidx[warm], split.labels[warm]
    parts = []
    for start in range(0, len(uidx), EVAL_BATCH):
        scores = score_fn(torch.as_tensor(uidx[start:start + EVAL_BATCH], device=DEVICE)).float()
        batch_labels = to_dense(labels[start:start + EVAL_BATCH]) > 0
        parts.append({k: v.cpu() for k, v in topk_metrics(scores, batch_labels, cfg.topks).items()})
    return pd.Series({m: torch.cat([p[m] for p in parts]).mean().item() for m in METRICS})

## 2. Training

In [5]:
deg_u = np.asarray(train_csr.sum(1)).ravel()
deg_i = np.asarray(train_csr.sum(0)).ravel()
adj_norm = to_torch_sparse(normalize_adj(train_csr), DEVICE)
SVD_CACHE = {}


def svd_factors(q, exact):
    """Rank-q SVD view of adj_norm, computed once per (q, exact)."""
    if (q, exact) not in SVD_CACHE:
        torch.manual_seed(cfg.seed)  # svd_lowrank is randomized
        SVD_CACHE[(q, exact)] = svd_view(adj_norm, q, exact)
    return SVD_CACHE[(q, exact)]


for q, exact in sorted({(v["svd_q"], v["svd_exact"]) for v in cfg.variants.values()}, reverse=True):
    singular_values = svd_factors(q, exact)[0].norm(dim=0)  # columns of U are orthonormal: each column of U·S has norm s
    print(f"q={q}, {'SVD chính xác' if exact else 'svd_lowrank'}: singular values "
          + ", ".join(fmt_num(s, 3) for s in singular_values.tolist()))

degree_stats = pd.DataFrame({
    "item / user": pd.Series(deg_u).describe(percentiles=[0.5, 0.9, 0.99]),
    "user / item": pd.Series(deg_i[deg_i > 0]).describe(percentiles=[0.5, 0.9, 0.99]),
}).T.drop(columns="std").rename(columns={"count": "số", "mean": "trung bình", "50%": "P50", "90%": "P90", "99%": "P99"})
display(fmt_table(degree_stats))
print(f"User chỉ có 1 cạnh: {fmt_pct((deg_u == 1).mean())}")

q=5, svd_lowrank: singular values 0,977, 0,724, 0,672, 0,651, 0,626
q=2, SVD chính xác: singular values 1, 0,804


,số,trung bình,min,P50,P90,P99,max
item / user,664.727,"2,6732",1,2,6,15,50
user / item,100,"17.769,1895",1.541,"16.282,5","30.345,6","46.969,71",62.880


User chỉ có 1 cạnh: 45,69%


In [6]:
MODEL_CLASSES = {"bpr": LightGCL, "softmax": LightGCLSoftmax}
assert all(spec["loss"] in MODEL_CLASSES for spec in cfg.variants.values()), f"loss must be among {list(MODEL_CLASSES)}"


def make_model(spec):
    factors = svd_factors(spec["svd_q"], spec["svd_exact"])
    torch.manual_seed(cfg.seed)  # every variant starts from the same embeddings
    return MODEL_CLASSES[spec["loss"]](
        n_users, n_items, cfg.embed_dim, *factors,
        train_csr, adj_norm, cfg.n_layers, cfg.temp, cfg.lambda_1, cfg.lambda_2, cfg.dropout,
    ).to(DEVICE)


def lightgcl_scorer(model):
    # evaluate() masks seen items for every model alike, so the model returns raw scores.
    return lambda uidx: model.predict(uidx, exclude_seen=False)


train_coo = train_csr.tocoo()
edge_users = torch.as_tensor(train_coo.row, dtype=torch.long, device=DEVICE)
edge_items = torch.as_tensor(train_coo.col, dtype=torch.long, device=DEVICE)
n_edges = len(edge_users)

# Rough peak of the user contrastive term: about three float32 copies of the batch × users matrix.
cl_gib = 3 * cfg.batch_size * n_users * 4 / 2**30
free_gib = torch.cuda.mem_get_info(DEVICE)[0] / 2**30 if DEVICE.type == "cuda" else float("nan")
print(f"{fmt_num((n_users + n_items) * cfg.embed_dim)} tham số mỗi model, {fmt_num(n_edges)} cạnh, "
      f"{fmt_num(-(-n_edges // cfg.batch_size))} bước mỗi epoch; phiên bản: {', '.join(cfg.variants)}")
print(f"Contrastive loss cần khoảng {cl_gib:.1f} GiB GPU (còn trống {free_gib:.1f} GiB)")
if cl_gib > free_gib:
    print("Cảnh báo: có thể hết bộ nhớ GPU, hãy giảm cfg.batch_size hoặc tắt kernel khác đang giữ GPU.")

42.548.928 tham số mỗi model, 1.776.919 cạnh, 1.736 bước mỗi epoch; phiên bản: Có negative sampling, q=5, Không negative sampling, q=5, Có negative sampling, q=2, Không negative sampling, q=2
Contrastive loss cần khoảng 7.6 GiB GPU (còn trống 11.8 GiB)


In [ ]:
def train_variant(label, spec):
    """Train one variant with early stopping on cfg.main_metric (validation); returns its best epoch and history."""
    model = make_model(spec)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=0)
    gen = torch.Generator(device=DEVICE).manual_seed(cfg.seed)  # same batch order for every variant
    score_fn = lightgcl_scorer(model)
    history, best, t_start = [], {"epoch": 0, "score": -np.inf, "state": None}, time.time()
    for epoch in range(1, cfg.epochs + 1):
        loss, loss_r, loss_s = train_epoch(
            model, optimizer, edge_users, edge_items, n_items, cfg.batch_size, gen,
            progress=partial(tqdm, desc=f"{label} · epoch {epoch}", leave=False),
        )
        if not np.isfinite(loss):
            raise FloatingPointError(f"[{label}] loss = {loss} ở epoch {epoch}")
        val = evaluate(score_fn, val_split)
        improved = val[cfg.main_metric] > best["score"]
        if improved:
            best = {"epoch": epoch, "score": val[cfg.main_metric], "state": copy.deepcopy(model.state_dict())}
        history.append({"epoch": epoch, "loss": loss, "loss_r": loss_r, "loss_s": loss_s, **val.to_dict()})
        print(f"[{label}] epoch {epoch:>2} | loss {loss:.4f} (rec {loss_r:.4f}, cl {loss_s:.4f}) | "
              f"val {cfg.main_metric} {val[cfg.main_metric]:.4f}{' *' if improved else ''} | {time.time() - t_start:.0f}s")
        if epoch - best["epoch"] >= cfg.patience:
            break
    return SimpleNamespace(best=best, history=pd.DataFrame(history).set_index("epoch"))


if cfg.retrain:
    run_name = f"lightgcl_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    run_dir = PROJECT_ROOT / "data" / "runs" / run_name
    run_dir.mkdir(parents=True)
    states, histories, results = {}, {}, {}
    for label, spec in cfg.variants.items():
        r = train_variant(label, spec)
        states[label], histories[label] = r.best["state"], r.history
        results[label] = {"best_epoch": r.best["epoch"], f"val {cfg.main_metric}": float(r.best["score"]),
                          "stop_epoch": len(r.history)}
        # Saved as soon as each variant finishes, so finished variants survive an interrupted run.
        torch.save({"states": states, "results": results, "user_ids": user_index.to_numpy()}, run_dir / "model.pt")
        pd.concat(histories, names=["variant", "epoch"]).to_csv(run_dir / "history.csv")
        cfg.variants[label]["checkpoint"] = (run_name, label)
        (run_dir / "config.json").write_text(json.dumps({**vars(cfg), "results": results}, ensure_ascii=False, indent=2))
        print(f"[{label}] đã lưu tham số epoch {r.best['epoch']} vào {run_dir.relative_to(PROJECT_ROOT)}/model.pt")
    display(fmt_table(pd.DataFrame(results).T))

## 3. Kết quả trên test

Thử nghiệm các trường hợp sau, đo bằng Recall@K và NDCG@K (K = 5, 10, 20) trên những user của tập test đã có trong graph train:

- **Có / không có negative sampling:** BPR với một item âm cho mỗi cặp dương, so với full softmax trên cả 100 cluster.
- **Hạng của SVD:** q = 5 (`torch.svd_lowrank`, như code gốc) và q = 2 (SVD chính xác).
- **So với hai baseline:**
  - **Popular:** xếp hotel cluster theo số user đã book hotel cluster đó trong giai đoạn train; mọi user nhận cùng một danh sách.
  - **History:** với mỗi user, các hotel cluster user đó đã book trong giai đoạn train đứng đầu, book càng nhiều lần thì xếp càng cao; các vị trí còn lại lấp theo thứ tự của Popular.

In [8]:
loaded = {}


def load_best(spec):
    """Variant model holding the parameters of its best validation epoch."""
    run, key = spec["checkpoint"]
    if run not in loaded:
        loaded[run] = torch.load(PROJECT_ROOT / "data" / "runs" / run / "model.pt", map_location="cpu", weights_only=False)
        assert np.array_equal(loaded[run]["user_ids"], user_index.to_numpy()), f"{run} was trained on another split"
    model = make_model(spec)
    model.load_state_dict(loaded[run]["states"][key])
    return model


scorers = {"Popular": pop_scores, "History": history_scores}
for label, spec in cfg.variants.items():
    scorers[label] = lightgcl_scorer(load_best(spec))
test_results = pd.DataFrame({name: evaluate(fn, test_split) for name, fn in scorers.items()}).T
display(fmt_table(test_results))

,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
Popular,"0,1526","0,2529","0,4066","0,1084","0,1436","0,1865"
History,"0,215","0,3252","0,4716","0,1604","0,1991","0,2401"
"Có negative sampling, q=5","0,1883","0,2894","0,4256","0,139","0,1748","0,2134"
"Không negative sampling, q=5","0,1895","0,2952","0,4384","0,14","0,1774","0,2179"
"Có negative sampling, q=2","0,2016","0,3147","0,4716","0,1471","0,1871","0,2314"
"Không negative sampling, q=2","0,2044","0,3231","0,483","0,1491","0,191","0,2362"
